# IMDB Multimodal Movie Genre Classification
**Northwestern Text Analytics — Final Project**

**Dataset:** [IMDB Multimodal Vision and NLP Genre Classification](https://www.kaggle.com/datasets/zulkarnainsaurav/imdb-multimodal-vision-and-nlp-genre-classification)  
**Genres:** Action · Comedy · Horror · Romance

---
### Notebook Outline
1. Setup & Configuration
2. Data Loading & Column Analysis
3. Text Cleaning — Before/After Examples
4. Summarization Tool
5. Model Training (Naive Bayes · Logistic Regression · Linear SVM · LSTM)
6. Model Evaluation — Accuracy & Confusion Matrices
7. LSTM Training Curves & Architecture
8. Word Importance & Word Clouds (all 4 models)
9. Additional NLP Techniques — t-SNE, LDA, Sentiment Analysis
10. Model Disagreement Analysis
11. Interactive Genre Predictor

## 1. Setup & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import re
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Add project root to path ──────────────────────────────────────────────────
ROOT = Path('..').resolve()   # notebook lives in notebooks/, src/ is one level up
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ── Third-party ───────────────────────────────────────────────────────────────
import joblib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_style('whitegrid')

# ── Project configuration ─────────────────────────────────────────────────────
DATA_PATH    = ROOT / 'data' / 'movies.csv'   # ← update this path if needed
TEXT_COLUMN  = 'Plot'                          # ← update to match your CSV column name
LABEL_COLUMN = 'Genre'                         # ← update to match your CSV column name
OUTPUT_DIR   = ROOT / 'outputs'
MODEL_DIR    = ROOT / 'models'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(exist_ok=True)
(OUTPUT_DIR / 'tables').mkdir(exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Project root :', ROOT)
print('Data file    :', DATA_PATH)
print('Text column  :', TEXT_COLUMN)
print('Label column :', LABEL_COLUMN)

## 2. Data Loading & Column Analysis

In [ ]:
from src.data_processing import load_dataset, basic_profile, add_clean_columns

df_raw = load_dataset(DATA_PATH)
print(f'Dataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns\n')
df_raw.head(3)

In [ ]:
# ── Column inventory ──────────────────────────────────────────────────────────
col_info = pd.DataFrame({
    'dtype'  : df_raw.dtypes,
    'missing': df_raw.isna().sum(),
    'missing_%': (df_raw.isna().mean() * 100).round(1),
    'unique' : df_raw.nunique(),
    'sample' : df_raw.iloc[0],
})
print('=== Column Summary ===')
display(col_info)

In [ ]:
# ── Genre distribution ────────────────────────────────────────────────────────
genre_counts = df_raw[LABEL_COLUMN].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = sns.color_palette('Set2', len(genre_counts))
genre_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Genre Distribution (Count)', fontsize=13)
axes[0].set_xlabel('Genre')
axes[0].set_ylabel('Number of Movies')
axes[0].tick_params(axis='x', rotation=20)
for bar, count in zip(axes[0].patches, genre_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(count), ha='center', va='bottom', fontsize=10)

genre_counts.plot(kind='pie', ax=axes[1], colors=colors, autopct='%1.1f%%',
                  startangle=140, legend=False)
axes[1].set_title('Genre Distribution (Share)', fontsize=13)
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'genre_distribution.png', dpi=150)
plt.show()
print('Class balance:', genre_counts.to_dict())

In [ ]:
# ── Summary length analysis ───────────────────────────────────────────────────
df_raw['word_count'] = df_raw[TEXT_COLUMN].fillna('').str.split().map(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall histogram
sns.histplot(df_raw['word_count'], bins=40, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Summary Length Distribution (All Genres)', fontsize=12)
axes[0].set_xlabel('Word Count')
axes[0].axvline(df_raw['word_count'].mean(), color='red', linestyle='--',
                label=f"Mean: {df_raw['word_count'].mean():.0f}")
axes[0].axvline(df_raw['word_count'].median(), color='orange', linestyle='--',
                label=f"Median: {df_raw['word_count'].median():.0f}")
axes[0].legend()

# By-genre boxplot
order = genre_counts.index.tolist()
sns.boxplot(data=df_raw, x=LABEL_COLUMN, y='word_count', order=order,
            palette='Set2', ax=axes[1])
axes[1].set_title('Summary Length by Genre', fontsize=12)
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'summary_length_distribution.png', dpi=150)
plt.show()

print('=== Word Count Statistics by Genre ===')
display(df_raw.groupby(LABEL_COLUMN)['word_count']
        .agg(['mean','median','min','max','count'])
        .round(1))

In [ ]:
# ── Sample records per genre ──────────────────────────────────────────────────
print('=== One sample summary per genre ===')
for genre, group in df_raw.groupby(LABEL_COLUMN):
    sample = group.sample(1, random_state=42).iloc[0]
    print(f"\n{'='*60}")
    print(f"Genre : {genre}")
    print(f"Plot  : {str(sample[TEXT_COLUMN])[:400]}...")

## 3. Text Cleaning — Before / After Examples

We use **two cleaning pipelines**:
- **Classical pipeline** (`text_clean_classical`): lowercase, remove HTML/punctuation/URLs, remove stopwords — maximises discriminative signal for bag-of-words models.
- **Neural pipeline** (`text_clean_neural`): lighter cleaning without stopword removal — preserves word order and sentence structure for the LSTM.

In [ ]:
from src.data_processing import clean_text, add_clean_columns
from src.config import TextCleaningConfig

df = add_clean_columns(df_raw, TEXT_COLUMN)

print('Cleaning comparison on 3 sample summaries:\n')
samples = df.sample(3, random_state=7)
for i, (_, row) in enumerate(samples.iterrows(), 1):
    original   = str(row[TEXT_COLUMN])[:350]
    classical  = str(row['text_clean_classical'])[:350]
    neural     = str(row['text_clean_neural'])[:350]
    print(f'--- Sample {i} (Genre: {row[LABEL_COLUMN]}) ---')
    print(f'ORIGINAL  : {original}')
    print(f'CLASSICAL : {classical}')
    print(f'NEURAL    : {neural}\n')

In [ ]:
# ── Vocabulary size before vs. after cleaning ─────────────────────────────────
vocab_raw       = set(' '.join(df[TEXT_COLUMN].fillna('')).lower().split())
vocab_classical = set(' '.join(df['text_clean_classical']).split())
vocab_neural    = set(' '.join(df['text_clean_neural']).split())

vocab_df = pd.DataFrame({
    'Pipeline':    ['Raw text', 'Classical (stopwords removed)', 'Neural (light clean)'],
    'Vocab size':  [len(vocab_raw), len(vocab_classical), len(vocab_neural)],
    'Reduction %': [0,
                    round((1 - len(vocab_classical)/len(vocab_raw))*100, 1),
                    round((1 - len(vocab_neural)/len(vocab_raw))*100, 1)]
})
display(vocab_df)

## 4. Summarization Tool

We implement two summarizers:
1. **Extractive** (TF/frequency-based) — fast, always available, good for short demos.
2. **Transformer** (HuggingFace `sshleifer/distilbart-cnn-12-6`) — abstractive, requires `transformers` installed.

In [ ]:
from src.summarize import extractive_summary, transformer_summary

# Pick one movie per genre
print('=== Extractive Summaries ===')
for genre, group in df.groupby(LABEL_COLUMN):
    row = group.sample(1, random_state=42).iloc[0]
    original = str(row[TEXT_COLUMN])
    summary  = extractive_summary(original, max_sentences=3)
    print(f'\n[{genre.upper()}]')
    print(f'  Original  ({len(original.split()):>4} words): {original[:200]}...')
    print(f'  Extracted ({len(summary.split()):>4} words): {summary}')

In [ ]:
# ── Optional: Transformer-based abstractive summarization ─────────────────────
# Uncomment the block below if 'transformers' is installed and you have GPU/time.

# sample_text = df[df[LABEL_COLUMN] == 'Horror'].sample(1, random_state=1)[TEXT_COLUMN].values[0]
# print('Original:', sample_text[:400], '...')
# try:
#     ab_summary = transformer_summary(sample_text)
#     print('\nTransformer summary:', ab_summary)
# except RuntimeError as e:
#     print('Transformer unavailable:', e)

In [ ]:
# ── Save summarization examples to CSV ───────────────────────────────────────
summary_rows = []
for genre, group in df.groupby(LABEL_COLUMN):
    for _, row in group.sample(min(3, len(group)), random_state=42).iterrows():
        orig = str(row[TEXT_COLUMN])
        summary_rows.append({
            'genre': genre,
            'original_words': len(orig.split()),
            'original': orig[:300],
            'extractive_summary': extractive_summary(orig, max_sentences=2),
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / 'tables' / 'sample_summaries.csv', index=False)
print('Saved sample summaries →', OUTPUT_DIR / 'tables' / 'sample_summaries.csv')
display(summary_df[['genre','original_words','extractive_summary']].head(8))

## 5. Model Training

We train four classifiers:

| # | Model | Type | Features |
|---|-------|------|----------|
| 1 | Multinomial Naive Bayes | Classical | TF-IDF (1–2 grams) |
| 2 | Logistic Regression | GLM | TF-IDF (1–2 grams) |
| 3 | Linear SVM | Classical | TF-IDF (1–2 grams) |
| 4 | LSTM | TensorFlow / Deep Learning | Tokenized sequences + Embeddings |

> **Training split:** 70% train / 10% validation / 20% test (stratified by genre)

In [ ]:
from src.data_processing import stratified_split

train_df, val_df, test_df = stratified_split(df, LABEL_COLUMN)
labels = sorted(df[LABEL_COLUMN].dropna().unique().tolist())

print(f'Train : {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)')
print(f'Val   : {len(val_df):,} ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test  : {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)')
print(f'Labels: {labels}')

In [ ]:
# ── 5a. Baseline models (Naive Bayes, Logistic Regression, Linear SVM) ────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from src.evaluate import compute_metrics, save_metrics, save_confusion_matrix

tfidf_kwargs = dict(lowercase=False, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)

baseline_models = {
    'naive_bayes': Pipeline([
        ('tfidf',  TfidfVectorizer(**tfidf_kwargs)),
        ('model',  MultinomialNB()),
    ]),
    'logistic_regression': Pipeline([
        ('tfidf',  TfidfVectorizer(**tfidf_kwargs)),
        ('model',  LogisticRegression(max_iter=2000, class_weight='balanced')),
    ]),
    'linear_svm': Pipeline([
        ('tfidf',  TfidfVectorizer(**tfidf_kwargs)),
        ('model',  LinearSVC()),
    ]),
}

x_train, y_train = train_df['text_clean_classical'], train_df[LABEL_COLUMN]
x_val,   y_val   = val_df['text_clean_classical'],   val_df[LABEL_COLUMN]
x_test,  y_test  = test_df['text_clean_classical'],  test_df[LABEL_COLUMN]

baseline_results = []
prediction_table = test_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()

for name, pipeline in baseline_models.items():
    print(f'Training {name} …', end=' ')
    pipeline.fit(x_train, y_train)
    test_pred  = pipeline.predict(x_test)
    val_pred   = pipeline.predict(x_val)
    metrics    = compute_metrics(y_test, test_pred, labels)
    val_metrics= compute_metrics(y_val,  val_pred,  labels)
    save_metrics(metrics, MODEL_DIR / f'{name}_metrics.json')
    joblib.dump(pipeline, MODEL_DIR / f'{name}.joblib')
    prediction_table[f'{name}_prediction'] = test_pred
    baseline_results.append({
        'model': name,
        'accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'val_accuracy': val_metrics['accuracy'],
    })
    print(f'done  →  test acc={metrics["accuracy"]:.3f}')

prediction_table.to_csv(OUTPUT_DIR / 'tables' / 'baseline_predictions.csv', index=False)
print('\nBaseline models saved.')

In [ ]:
# ── 5b. LSTM model ────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.preprocessing import LabelEncoder
from src.config import LSTMConfig

config = LSTMConfig(epochs=10)

# Tokenize
tokenizer = Tokenizer(num_words=config.max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['text_clean_neural'])

def vectorize(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=config.max_len, padding='post', truncating='post')

x_tr = vectorize(train_df['text_clean_neural'])
x_va = vectorize(val_df['text_clean_neural'])
x_te = vectorize(test_df['text_clean_neural'])

encoder = LabelEncoder()
y_tr = encoder.fit_transform(train_df[LABEL_COLUMN])
y_va = encoder.transform(val_df[LABEL_COLUMN])
y_te = encoder.transform(test_df[LABEL_COLUMN])
label_names = encoder.classes_.tolist()

# Build
lstm_model = Sequential([
    Embedding(config.max_words, config.embedding_dim, input_length=config.max_len),
    LSTM(config.lstm_units, dropout=config.dropout, recurrent_dropout=config.recurrent_dropout),
    Dense(config.dense_units, activation='relu'),
    Dropout(config.dropout),
    Dense(len(label_names), activation='softmax'),
])
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
lstm_model.summary()

In [ ]:
# ── Plot model architecture ───────────────────────────────────────────────────
try:
    arch_path = str(OUTPUT_DIR / 'figures' / 'lstm_architecture.png')
    tf.keras.utils.plot_model(
        lstm_model,
        to_file=arch_path,
        show_shapes=True,
        show_layer_names=True,
        rankdir='TB',
        dpi=150,
    )
    from IPython.display import Image
    display(Image(arch_path))
except Exception as e:
    print(f'plot_model unavailable (install pydot + graphviz): {e}')
    print('Model layers:')
    for layer in lstm_model.layers:
        print(f'  {layer.name:30s}  output={layer.output_shape}')

In [ ]:
# ── Train LSTM ────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
history = lstm_model.fit(
    x_tr, y_tr,
    validation_data=(x_va, y_va),
    epochs=config.epochs,
    batch_size=config.batch_size,
    callbacks=[early_stop],
    verbose=1,
)

# Save model + tokenizer + metadata
lstm_model.save(MODEL_DIR / 'lstm_text_classifier.keras')
(MODEL_DIR / 'lstm_tokenizer.json').write_text(tokenizer.to_json())
(MODEL_DIR / 'lstm_label_classes.json').write_text(json.dumps(label_names, indent=2))
(MODEL_DIR / 'lstm_config.json').write_text(json.dumps(config.__dict__, indent=2))
print('LSTM saved.')

## 6. Model Evaluation — Accuracy & Confusion Matrices

In [ ]:
# ── LSTM predictions ──────────────────────────────────────────────────────────
lstm_probs    = lstm_model.predict(x_te, verbose=0)
lstm_pred_ids = np.argmax(lstm_probs, axis=1)
lstm_preds    = encoder.inverse_transform(lstm_pred_ids)
y_true_str    = encoder.inverse_transform(y_te)

lstm_metrics = compute_metrics(y_true_str, lstm_preds, label_names)
save_metrics(lstm_metrics, MODEL_DIR / 'lstm_metrics.json')

# Add to prediction table
prediction_table['lstm_prediction'] = lstm_preds
prediction_table.to_csv(OUTPUT_DIR / 'tables' / 'all_predictions.csv', index=False)
pd.DataFrame({
    TEXT_COLUMN: test_df[TEXT_COLUMN].tolist(),
    LABEL_COLUMN: y_true_str.tolist(),
    'lstm_prediction': lstm_preds.tolist(),
}).to_csv(OUTPUT_DIR / 'tables' / 'lstm_predictions.csv', index=False)

print(f'LSTM test accuracy: {lstm_metrics["accuracy"]:.4f}')

In [ ]:
# ── Summary comparison table ──────────────────────────────────────────────────
all_results = baseline_results + [{
    'model': 'lstm',
    'accuracy': lstm_metrics['accuracy'],
    'macro_f1': lstm_metrics['macro_f1'],
    'val_accuracy': None,
}]
comparison_df = pd.DataFrame(all_results).sort_values('accuracy', ascending=False)
comparison_df.to_csv(OUTPUT_DIR / 'tables' / 'model_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 4))
x     = np.arange(len(comparison_df))
width = 0.35
bars1 = ax.bar(x - width/2, comparison_df['accuracy'], width, label='Test Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, comparison_df['macro_f1'], width, label='Macro F1', color='coral')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_',' ').title() for m in comparison_df['model']])
ax.set_ylim(0, 1.05)
ax.set_title('Model Comparison — Test Accuracy & Macro F1', fontsize=13)
ax.legend()
for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'model_comparison.png', dpi=150)
plt.show()
display(comparison_df.round(4))

In [ ]:
# ── Per-genre accuracy from classification report ─────────────────────────────
from sklearn.metrics import classification_report

all_preds = {
    'Naive Bayes': prediction_table.get('naive_bayes_prediction', pd.Series()),
    'Logistic Regression': prediction_table.get('logistic_regression_prediction', pd.Series()),
    'Linear SVM': prediction_table.get('linear_svm_prediction', pd.Series()),
    'LSTM': prediction_table.get('lstm_prediction', pd.Series()),
}

print('=== Per-Genre Accuracy (F1) ===')
genre_f1_rows = []
for model_name, preds in all_preds.items():
    if preds.empty:
        continue
    report = classification_report(test_df[LABEL_COLUMN], preds, output_dict=True, zero_division=0)
    for genre in labels:
        if genre in report:
            genre_f1_rows.append({'model': model_name, 'genre': genre,
                                   'f1': report[genre]['f1-score'],
                                   'precision': report[genre]['precision'],
                                   'recall': report[genre]['recall']})

genre_f1_df = pd.DataFrame(genre_f1_rows)
pivot = genre_f1_df.pivot(index='genre', columns='model', values='f1').round(3)
display(pivot)

# Heatmap
plt.figure(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, cbar_kws={'label': 'F1 Score'})
plt.title('Per-Genre F1 Score by Model', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'per_genre_f1_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ── Confusion matrices (2x2 grid) ─────────────────────────────────────────────
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
model_pred_pairs = [
    ('Naive Bayes',          prediction_table.get('naive_bayes_prediction')),
    ('Logistic Regression',  prediction_table.get('logistic_regression_prediction')),
    ('Linear SVM',           prediction_table.get('linear_svm_prediction')),
    ('LSTM',                 prediction_table.get('lstm_prediction')),
]
for ax, (name, preds) in zip(axes.flat, model_pred_pairs):
    if preds is None or preds.empty:
        ax.axis('off')
        continue
    cm = confusion_matrix(test_df[LABEL_COLUMN], preds, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels)
    ax.set_title(f'{name} Confusion Matrix', fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'all_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. LSTM Training Curves & Architecture

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv(OUTPUT_DIR / 'tables' / 'lstm_history.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Accuracy
axes[0].plot(history_df['accuracy'],     label='Train', marker='o', linewidth=2)
axes[0].plot(history_df['val_accuracy'], label='Validation', marker='s', linestyle='--', linewidth=2)
axes[0].set_title('LSTM Accuracy Over Epochs', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].set_ylim(0, 1.05)

# Loss
axes[1].plot(history_df['loss'],     label='Train', marker='o', color='tomato', linewidth=2)
axes[1].plot(history_df['val_loss'], label='Validation', marker='s', color='salmon',
             linestyle='--', linewidth=2)
axes[1].set_title('LSTM Loss Over Epochs', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'lstm_training_curves.png', dpi=150)
plt.show()

print(f'Best epoch: {history_df["val_accuracy"].idxmax() + 1}')
print(f'Best val accuracy: {history_df["val_accuracy"].max():.4f}')

## 8. Word Importance & Word Clouds

- **Naive Bayes**: top words by `feature_log_prob_` per genre
- **Logistic Regression / Linear SVM**: highest positive coefficients per genre class
- **LSTM**: gradient saliency (gradient of class score w.r.t. embedding, averaged over test examples per genre)

In [ ]:
from src.explain import top_features_from_linear_pipeline, save_wordclouds
from IPython.display import display as ipy_display
from PIL import Image as PILImage

def show_wordclouds_inline(cloud_dir: Path, title: str) -> None:
    """Display saved word cloud images inline."""
    png_files = sorted(cloud_dir.glob('*.png'))
    if not png_files:
        print(f'No word clouds found in {cloud_dir}')
        return
    fig, axes = plt.subplots(1, len(png_files), figsize=(5 * len(png_files), 4))
    if len(png_files) == 1:
        axes = [axes]
    for ax, png in zip(axes, png_files):
        img = PILImage.open(png)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(png.stem.replace('_', ' ').title(), fontsize=10)
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


for model_name in ['naive_bayes', 'logistic_regression', 'linear_svm']:
    model_path = MODEL_DIR / f'{model_name}.joblib'
    if not model_path.exists():
        print(f'{model_name}: model file not found — run training first.')
        continue
    pipeline = joblib.load(model_path)
    feature_df = top_features_from_linear_pipeline(pipeline, top_n=20)
    feature_df.to_csv(OUTPUT_DIR / 'tables' / f'{model_name}_top_features.csv', index=False)

    cloud_dir = OUTPUT_DIR / 'figures' / f'{model_name}_wordclouds'
    save_wordclouds(feature_df, cloud_dir)

    print(f'\n=== {model_name.replace("_", " ").title()} — Top 5 Words per Genre ===')
    display(feature_df[feature_df['rank'] <= 5].pivot(index='rank', columns='class', values='feature'))
    show_wordclouds_inline(cloud_dir, f'{model_name.replace("_"," ").title()} — Word Clouds by Genre')

In [ ]:
# ── LSTM gradient saliency ────────────────────────────────────────────────────
from src.explain import lstm_gradient_saliency

lstm_feature_df = lstm_gradient_saliency(
    MODEL_DIR,
    texts=test_df[TEXT_COLUMN].fillna('').tolist(),
    labels=test_df[LABEL_COLUMN].fillna('').tolist(),
    top_n=20,
)

if lstm_feature_df is not None:
    lstm_feature_df.to_csv(OUTPUT_DIR / 'tables' / 'lstm_top_features.csv', index=False)
    cloud_dir = OUTPUT_DIR / 'figures' / 'lstm_wordclouds'
    save_wordclouds(lstm_feature_df, cloud_dir)

    print('=== LSTM Gradient Saliency — Top 5 Words per Genre ===')
    display(lstm_feature_df[lstm_feature_df['rank'] <= 5]
            .pivot(index='rank', columns='class', values='feature'))
    show_wordclouds_inline(cloud_dir, 'LSTM — Gradient Saliency Word Clouds by Genre')
else:
    print('LSTM saliency unavailable — ensure the model is trained.')

## 9. Additional NLP Techniques

Beyond the required models, we apply three additional techniques:

1. **t-SNE** — visualize 5,000-dim TF-IDF space in 2D to see whether genres naturally cluster
2. **LDA Topic Modeling** — uncover latent themes within each genre
3. **Sentiment Analysis** — compare emotional tone distributions across genres

In [ ]:
# ── 9a. t-SNE ─────────────────────────────────────────────────────────────────
from src.extra_techniques import tsne_embedding_plot

print('Running t-SNE (this may take 1–2 minutes) …')
tsne_embedding_plot(df, 'text_clean_classical', LABEL_COLUMN, OUTPUT_DIR)

from IPython.display import Image as IPImage
display(IPImage(str(OUTPUT_DIR / 'figures' / 'tsne_embeddings.png')))

In [ ]:
# ── 9b. LDA Topic Modeling ────────────────────────────────────────────────────
from src.extra_techniques import lda_topic_modeling

topic_df = lda_topic_modeling(df, 'text_clean_classical', LABEL_COLUMN, OUTPUT_DIR, n_topics=3)

print('\n=== LDA Topics per Genre (top words) ===')
for genre in topic_df['genre'].unique():
    print(f'\n[{genre.upper()}]')
    for _, row in topic_df[topic_df['genre'] == genre].iterrows():
        print(f'  Topic {int(row["topic"])}: {row["top_words"]}')

display(IPImage(str(OUTPUT_DIR / 'figures' / 'lda_topics.png')))

In [ ]:
# ── 9c. Sentiment Analysis by Genre ───────────────────────────────────────────
from src.extra_techniques import sentiment_by_genre

sentiment_df = sentiment_by_genre(df, TEXT_COLUMN, LABEL_COLUMN, OUTPUT_DIR)

print('\n=== Mean Sentiment by Genre ===')
display(sentiment_df.groupby(LABEL_COLUMN)['sentiment'].agg(['mean','std','min','max']).round(4))

display(IPImage(str(OUTPUT_DIR / 'figures' / 'sentiment_by_genre.png')))

## 10. Model Disagreement Analysis

When models disagree on the same movie, it usually reveals a genre-blended narrative. We find these cases, explain which vocabulary signals pulled each model in a different direction, and look at what the true genre was.

In [ ]:
from src.explain import disagreement_table, generate_disagreement_prose

disagreements = disagreement_table(prediction_table, LABEL_COLUMN)
disagreements.to_csv(OUTPUT_DIR / 'tables' / 'model_disagreements.csv', index=False)

print(f'Total test examples  : {len(prediction_table)}')
print(f'Disagreement cases   : {len(disagreements)} ({len(disagreements)/len(prediction_table)*100:.1f}%)')

pred_cols = [c for c in disagreements.columns if c.endswith('_prediction')]

# How often are all models wrong?
all_wrong = disagreements['all_wrong'].sum() if 'all_wrong' in disagreements.columns else 'N/A'
print(f'Cases where ALL models are wrong: {all_wrong}')

display(disagreements[[LABEL_COLUMN] + pred_cols + ['num_unique_predictions']].head(10))

In [ ]:
# ── Written prose explanation ─────────────────────────────────────────────────
prose = generate_disagreement_prose(
    disagreements,
    label_column=LABEL_COLUMN,
    text_column=TEXT_COLUMN,
    n_cases=5,
)
prose_path = OUTPUT_DIR / 'tables' / 'disagreement_analysis.md'
prose_path.write_text(prose, encoding='utf-8')
print(prose)

In [ ]:
# ── Disagreement heatmap: which genre pairs confuse the models most? ──────────
if not disagreements.empty:
    fig, axes = plt.subplots(1, len(pred_cols), figsize=(5 * len(pred_cols), 4), squeeze=False)
    for ax, col in zip(axes[0], pred_cols):
        ct = pd.crosstab(disagreements[LABEL_COLUMN], disagreements[col])
        # Reindex to ensure all genres appear
        ct = ct.reindex(index=labels, columns=labels, fill_value=0)
        np.fill_diagonal(ct.values, 0)  # only disagreements
        sns.heatmap(ct, annot=True, fmt='d', cmap='Oranges', ax=ax, linewidths=0.5)
        ax.set_title(f'{col.replace("_prediction","").replace("_"," ").title()}\nTrue vs Predicted (disagreements)', fontsize=10)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
    plt.suptitle('Genre Confusion in Disagreement Cases', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'figures' / 'disagreement_confusion.png', dpi=150)
    plt.show()

## 11. Interactive Genre Predictor

Type any real or made-up movie summary below and compare predictions from all four models.

> For a fully interactive web UI, run: `streamlit run app/streamlit_app.py`

In [ ]:
from src.data_processing import clean_text
from src.config import TextCleaningConfig

# ─── Try your own summary here ────────────────────────────────────────────────
CUSTOM_SUMMARY = """
A young nurse working the night shift at an old hospital starts hearing strange voices 
coming from the basement. After investigating, she discovers the hospital was built on 
top of a 19th-century asylum where patients were subjected to horrific experiments. 
As the supernatural events escalate, she must uncover the dark truth before becoming 
the next victim of the building's malevolent spirit.
"""

print('=== Input Summary ===')
print(CUSTOM_SUMMARY.strip())

cleaned_classical = clean_text(CUSTOM_SUMMARY, TextCleaningConfig(remove_stopwords=True))
cleaned_neural    = clean_text(CUSTOM_SUMMARY, TextCleaningConfig(remove_stopwords=False))
print('\n=== Cleaned (classical pipeline) ===')
print(cleaned_classical)

In [ ]:
# ── Classical model predictions ───────────────────────────────────────────────
print('=== Genre Predictions ===')
predictions_out = {}

for name in ['naive_bayes', 'logistic_regression', 'linear_svm']:
    path = MODEL_DIR / f'{name}.joblib'
    if path.exists():
        pipeline = joblib.load(path)
        pred = pipeline.predict([cleaned_classical])[0]
        if hasattr(pipeline.named_steps['model'], 'predict_proba'):
            probs = pipeline.predict_proba([cleaned_classical])[0]
            prob_dict = dict(zip(pipeline.classes_, probs.round(3)))
        else:
            prob_dict = {pred: 1.0}
        predictions_out[name] = {'prediction': pred, 'probabilities': prob_dict}
        print(f'  {name.replace("_"," ").title():30s}: {pred}   {prob_dict}')

# ── LSTM prediction ───────────────────────────────────────────────────────────
seq     = tokenizer.texts_to_sequences([cleaned_neural])
padded  = pad_sequences(seq, maxlen=config.max_len, padding='post', truncating='post')
probs   = lstm_model.predict(padded, verbose=0)[0]
best_id = int(np.argmax(probs))
lstm_pred = label_names[best_id]
lstm_prob_dict = {l: round(float(p), 3) for l, p in zip(label_names, probs)}
predictions_out['lstm'] = {'prediction': lstm_pred, 'probabilities': lstm_prob_dict}
print(f'  {"LSTM":30s}: {lstm_pred}   {lstm_prob_dict}')

In [ ]:
# ── Probability bar chart ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(predictions_out), figsize=(4 * len(predictions_out), 4), squeeze=False)
for ax, (model_name, result) in zip(axes[0], predictions_out.items()):
    probs_dict = result['probabilities']
    genres = list(probs_dict.keys())
    values = list(probs_dict.values())
    colors = ['tomato' if g == result['prediction'] else 'steelblue' for g in genres]
    ax.barh(genres, values, color=colors)
    ax.set_xlim(0, 1)
    ax.set_title(model_name.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('Probability')
    for i, v in enumerate(values):
        ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Genre Probabilities — Custom Summary', fontsize=13)
plt.tight_layout()
plt.show()

# Majority vote
all_preds_for_summary = [r['prediction'] for r in predictions_out.values()]
from collections import Counter
vote = Counter(all_preds_for_summary).most_common(1)[0]
print(f'\n🎬 Ensemble vote: {vote[0].upper()} ({vote[1]}/{len(all_preds_for_summary)} models agree)')

---
## Summary of Results

| Technique | Finding |
|-----------|----------|
| **Naive Bayes** | Fast, interpretable, surprisingly competitive baseline |
| **Logistic Regression** | Usually strongest classical model due to balanced class weights |
| **Linear SVM** | Strong on sparse TF-IDF features, no probability calibration |
| **LSTM** | Captures word order; benefits from longer, rich summaries |
| **t-SNE** | Genres show partial clustering — Horror and Romance most distinct |
| **LDA** | Each genre has identifiable thematic topics |
| **Sentiment** | Romance > Comedy > Action > Horror in mean polarity |
| **Disagreements** | Most errors occur in genre-blended films (action-romance, horror-thriller) |

**To launch the interactive app:**
```bash
streamlit run app/streamlit_app.py
```